# Clase 8: Redes convolucionales. Laboratorio

Objetivo: escribir dos redes como clases de PyTorch y entrenarlas para reconocer los dígitos de MNIST.

Este laboratorio tiene tres archivos:

| Archivo | Qué es |
|---|---|
| `modelos.py` | Las redes y el paso de entrenamiento. Es el único archivo que se edita. |
| `utilidades.py` | Funciones de apoyo: cargar los datos, entrenar, graficar. No hace falta modificarlo. |
| `lab.ipynb` | Este notebook: importa las clases de `modelos.py`, las verifica y las entrena. |

Protocolo: editar una clase en `modelos.py`, guardar el archivo y volver a ejecutar la celda de verificación aquí. Las celdas de verificación indican si la red es correcta o qué está mal.

> Si el notebook no ve los cambios en `modelos.py`: guardar el archivo y volver a ejecutar la celda. Si aun así no los ve, reiniciar el kernel (*Kernel → Restart*) y ejecutar las celdas desde el principio.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import matplotlib.pyplot as plt

import modelos
from utilidades import *

X_train, y_train, X_val, y_val, X_test, y_test = load_mnist()
print("entrenamiento:", X_train.shape, y_train.shape)
print("validación:   ", X_val.shape)
print("prueba:       ", X_test.shape)

Las imágenes tienen forma $(N, C, H, W)$: $N$ imágenes, de $C = 1$ canal (escala de grises), de $28 \times 28$ píxeles, con valores entre 0 y 1. Estas son las primeras diez:

In [ ]:
fig, axs = plt.subplots(1, 10, figsize=(12, 1.6))
for i, ax in enumerate(axs):
    ax.imshow(X_train[i, 0], cmap="gray_r")
    ax.set_title(int(y_train[i]))
    ax.axis("off")
plt.show()

# Parte 1: `MyMLP`

## Ejercicio 1

Abrir `modelos.py` y completar la clase `MyMLP`: una red densa con una capa oculta de 128 unidades.

- En `__init__`, declarar `self.hidden` (784 → 128) y `self.output` (128 → 10) con `nn.Linear`.
- En `forward`, aplanar la imagen, pasarla por la capa oculta con ReLU y luego por la capa de salida. Sin softmax: `nn.CrossEntropyLoss` lo aplica.

Es la red de la sección 2.1 de las notas. Al terminar, guardar el archivo y ejecutar la celda de abajo.

<details><summary>Pista</summary>

```python
def __init__(self):
    super().__init__()
    self.hidden = nn.Linear(784, 128)
    self.output = nn.Linear(128, 10)
```
En `forward`: `x = x.flatten(start_dim=1)`, después `x = torch.relu(self.hidden(x))` y `return self.output(x)`.
</details>

In [ ]:
torch.manual_seed(0)
mlp = modelos.MyMLP()
shape = shape_table(mlp)

print()
verify(shape == (1, 10), "la salida es (1, 10): un score por dígito",
       f"la salida debería ser (1, 10), y es {shape}")
verify(count_parameters(mlp) == 101770, "101770 parámetros, como en las notas",
       f"deberían ser 101770 parámetros, y son {count_parameters(mlp)}")

La tabla de arriba es un forward de prueba: la red recibe una imagen inventada de forma $(1, 1, 28, 28)$ y, para cada capa, se imprime la forma de lo que entra, de lo que sale y su número de parámetros.

Pregunta. El entrenamiento usa 50 000 imágenes y mini-batches de 64. ¿Cuántos pasos de descenso por gradiente tiene una época? Calcularlo antes de entrenar y compararlo con lo que imprime `train`.

## Ejercicio 1b

Completar la función `train_step` en `modelos.py`: un paso de descenso por gradiente sobre un mini-batch. Es el corazón del ciclo de entrenamiento de la sección 2.2 de las notas, y son cinco líneas:

1. borrar los gradientes del paso anterior;
2. forward: los scores de la red para el mini-batch;
3. la pérdida;
4. backward: los gradientes;
5. el paso del optimizador.

Devolver la pérdida como número (`.item()`).

<details><summary>Pista</summary>

```python
optimizer.zero_grad()
scores = model(X_batch)
loss = loss_fn(scores, y_batch)
loss.backward()
optimizer.step()
return loss.item()
```
</details>

Pregunta. ¿Qué pasa si se omite `optimizer.zero_grad()`? Cuando la función pase la verificación, borrar esa línea y volver a ejecutar la celda.

In [ ]:
torch.manual_seed(0)
check_train_step(modelos.train_step, modelos.MyMLP(), X_train[:64], y_train[:64])

## Entrenar

En cada época, la función `train` recorre los datos en mini-batches, llama a `train_step` (de `modelos.py`) con cada uno y, al terminar la época, mide la exactitud sobre los datos de validación.

Idealmente, leer el código de `train` y `train_step` para entender qué hacen. Después, ejecutar la celda de abajo para entrenar la red y graficar la pérdida y la exactitud.

In [ ]:
torch.manual_seed(0)
mlp = modelos.MyMLP()
history_mlp = train(mlp, X_train, y_train, X_val, y_val, num_epochs=3, lr=0.1)

In [ ]:
verify(history_mlp is not None and history_mlp["val_accuracy"][-1] > 0.93,
       "la red aprendió: más del 93% de exactitud de validación",
       "la exactitud de validación debería pasar del 93%. Revisar el forward.")

Las predicciones de la red sobre algunas imágenes de validación, con la probabilidad que les asigna (softmax). Abajo, algunas que clasifica mal:

In [ ]:
show_predictions(mlp, X_val, y_val, n=12)
show_predictions(mlp, X_val, y_val, n=12, errors=True)

# Parte 2: `MyCNN` (~40 min)

## Ejercicio 2a (en papel, antes de programar)

`MyCNN` tiene esta arquitectura (la de la sección 3.7 de las notas):

```
Conv2d(1, 8, kernel_size=3, padding=1) -> ReLU -> MaxPool2d(2)
Conv2d(8, 16, kernel_size=3, padding=1) -> ReLU -> MaxPool2d(2)
aplanar -> Linear(?, 10)
```

Completar en papel la forma de la salida de cada capa, empezando por $1 \times 28 \times 28$. ¿Cuántas entradas tiene la capa densa? ¿Cuántos parámetros tiene cada capa?

| Capa | Forma de la salida | Parámetros |
|---|---|---|
| entrada | $1 \times 28 \times 28$ | — |
| conv1 + ReLU | | |
| pool | | |
| conv2 + ReLU | | |
| pool | | |
| aplanar | | |
| densa | $10$ | |

## Ejercicio 2b

Completar la clase `MyCNN` en `modelos.py`. Usar exactamente los nombres `self.conv1`, `self.conv2`, `self.pool` y `self.dense`. Guardar y ejecutar la celda de abajo.

<details><summary>Pista</summary>

```python
self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
self.pool = nn.MaxPool2d(kernel_size=2)
self.dense = nn.Linear(16 * 7 * 7, 10)
```
En `forward`: `x = self.pool(torch.relu(self.conv1(x)))`, lo mismo con `conv2`, después `x = x.flatten(start_dim=1)` y `return self.dense(x)`.
</details>

In [ ]:
torch.manual_seed(0)
cnn = modelos.MyCNN()
shape = shape_table(cnn)

print()
verify(shape == (1, 10), "la salida es (1, 10)",
       f"la salida debería ser (1, 10), y es {shape}")
verify(count_parameters(cnn) == 9098, "9098 parámetros: comparar con la tabla en papel",
       f"deberían ser 9098 parámetros, y son {count_parameters(cnn)}. Comparar capa por capa con la tabla en papel")

## Entrenar

Con el mismo código y los mismos datos que `MyMLP`. Tarda un poco más, porque cada convolución hace muchas operaciones.

In [ ]:
torch.manual_seed(0)
cnn = modelos.MyCNN()
history_cnn = train(cnn, X_train, y_train, X_val, y_val, num_epochs=3, lr=0.1)

In [ ]:
verify(history_cnn is not None and history_cnn["val_accuracy"][-1] > 0.96,
       "la red aprendió: más del 96% de exactitud de validación",
       "la exactitud de validación debería pasar del 96%. Revisar el forward.")

## MLP contra CNN

In [ ]:
plot_histories({"MyMLP": history_mlp, "MyCNN": history_cnn})

if history_mlp is not None and history_cnn is not None:
    print(f"{'red':8} {'parámetros':>11} {'validación':>11}")
    print(f"{'MyMLP':8} {count_parameters(mlp):>11} {history_mlp['val_accuracy'][-1]:>11.3f}")
    print(f"{'MyCNN':8} {count_parameters(cnn):>11} {history_cnn['val_accuracy'][-1]:>11.3f}")

Preguntas.

1. ¿Cuál de las dos redes tiene más parámetros? ¿Cuál es más precisa?
2. ¿Por qué la CNN necesita tan pocos parámetros?
3. La CNN tarda más en entrenar aunque tiene menos parámetros. ¿Por qué?

## Lo que aprendió la red

Arriba, los 8 filtros de $3 \times 3$ de `conv1` (oscuro: peso positivo; claro: negativo). Abajo, el mapa de activación de cada filtro para una imagen de validación. ¿Algún filtro parece detectar bordes? ¿De qué dirección?

In [ ]:
plot_filters(cnn, X_val[0])

Y algunos errores de la CNN. ¿Son los mismos tipos de errores que los del MLP?

In [ ]:
show_predictions(cnn, X_val, y_val, n=12, errors=True)

## La exactitud de prueba

Los datos de prueba se usan una sola vez, al final, para reportar el resultado del modelo escogido. Si los usáramos para comparar o ajustar modelos, el resultado dejaría de ser una estimación honesta de cómo funciona la red con datos nuevos.

In [ ]:
if history_cnn is not None:
    print(f"Exactitud de prueba de MyCNN: {accuracy(cnn, X_test, y_test):.3f}")

# Parte 3: Reto

Completar la clase `MyCNNVariant` en `modelos.py`: copiar `MyCNN` y cambiar algo de la arquitectura. Por ejemplo:

- más filtros (16 y 32 en lugar de 8 y 16);
- filtros de $5 \times 5$ (con `padding=2`);
- un tercer bloque de convolución;
- una capa densa oculta antes de la salida.

Antes de ejecutar, predecir en papel cuántos parámetros va a tener. Después, entrenar y comparar con `MyCNN`. ¿El cambio ayudó? ¿Cuánto costó en parámetros y en tiempo?

In [ ]:
if implemented(modelos.MyCNNVariant):
    torch.manual_seed(0)
    variant = modelos.MyCNNVariant()
    shape_table(variant)
    print()
    history_variant = train(variant, X_train, y_train, X_val, y_val, num_epochs=3, lr=0.1)
    plot_histories({"MyCNN": history_cnn, "Variant": history_variant})
else:
    print("MyCNNVariant todavía no está completa en modelos.py (es el reto para la casa).")

## Discusión

Copiar la salida de esta celda en la discusión de Classroom, junto con una breve explicación de los cambios hechos y por qué. Después, comentar la publicación de otra pareja: ¿su cambio funcionó mejor o peor que el propio?

In [ ]:
if implemented(modelos.MyCNNVariant):
    summary("MyCNNVariant", variant, history_variant)
else:
    summary("MyCNN", cnn, history_cnn)